# 01. Search Papers

OpenAlex API로 키워드 검색 → `data/papers_raw.csv`로 저장.

**OpenAlex**는 무인증·무료·일일 100k 호출이고 모든 분야 + 인용 데이터를 줍니다. 가장 부담 없이 시작할 수 있습니다.

추후 arXiv, Semantic Scholar로 확장 가능 — 같은 노트북 안에 별도 셀로 추가하세요.

In [ ]:
!pip install -r ../../../requirements.txt

  Using cached tabulate-0.10.0-py3-none-any.whl.metadata (40 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached patsy-1.0.2-py2.py3-none-any.whl.metadata (3.6 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 50.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 27.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 37.0 MB/s  0:00:00m0:00:01
Using cached urllib3-2.6.3-py3-none-any.whl (131 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 50.6 MB/s  0:00:00
   ━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.

In [1]:
import os
import time
import requests
import pandas as pd
from pathlib import Path
from typing import Optional

DATA_DIR = Path('../data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

# 본인 이메일을 넣으면 OpenAlex가 'polite pool'에 넣어 더 안정적인 응답을 줍니다.
MAILTO = os.environ.get('OPENALEX_MAILTO', 'student@example.com')


ModuleNotFoundError: No module named 'requests'

## 검색 파라미터

`topic_scoping(rr)`에서 받은 키워드와 연도 범위를 여기에 넣습니다.

In [ ]:
QUERY = 'agent architectures large language models'   # ← 본인 키워드로 교체
FROM_YEAR = 2020
TO_YEAR = 2026
MAX_RESULTS = 50

## 검색 함수

기본 호출 외에 **선택 옵션**으로 학생이 본인 주제에 정밀도를 맞출 수 있습니다.

| 옵션 | 의미 | 사용 예 |
|---|---|---|
| `search_field='title_and_abstract'` | 제목·abstract만 검색 (저자명·기관 매칭 노이즈 ↓) | 좁은 주제 정밀 검색 |
| `language='en'` | 영어 논문만 | 국제 학술지 위주 |
| `language='ko'` | 한국어 논문만 | 국내 사례·정책 |
| `country_code='KR'` | 저자 기관이 한국 | 국내 연구 동향 |
| `max_retries=5` | 네트워크 재시도 횟수 | 강의실 Wi-Fi 불안정 시 |

기본 호출(아무 옵션 없이)은 가장 넓은 검색 — 첫 시도 권장.

In [ ]:
def search_openalex(
    query: str,
    from_year: int,
    to_year: int,
    max_results: int = 50,
    mailto: str = MAILTO,
    *,
    search_field: str = 'default',          # 'default' | 'title' | 'title_and_abstract'
    language: Optional[str] = None,          # 'en', 'ko' 등 ISO-639-1
    country_code: Optional[str] = None,      # 'KR', 'US' 등 (저자 기관 기준)
    max_retries: int = 3,
    retry_backoff: float = 2.0,
) -> pd.DataFrame:
    """OpenAlex Works 검색 — 페이지네이션 + 재시도 + 선택적 필터 포함.

    학생이 자기 주제에 적용할 때 한 줄 옵션으로 정밀도·범위·언어를 조정 가능.
    """
    base_url = 'https://api.openalex.org/works'

    # 1) 검색 정밀도 — search_field에 따라 query를 다른 파라미터로 보냄
    extra_filter = None
    base_params = {'mailto': mailto}
    if search_field == 'title':
        extra_filter = f'title.search:{query}'
    elif search_field == 'title_and_abstract':
        extra_filter = f'title_and_abstract.search:{query}'
    else:
        base_params['search'] = query

    # 2) 필터 조합 (date 필수 + 옵션)
    filters = [
        f'from_publication_date:{from_year}-01-01',
        f'to_publication_date:{to_year}-12-31',
    ]
    if language:
        filters.append(f'language:{language}')
    if country_code:
        filters.append(f'authorships.institutions.country_code:{country_code.lower()}')
    if extra_filter:
        filters.append(extra_filter)
    base_params['filter'] = ','.join(filters)

    # 3) 페이지네이션 + 재시도
    rows, cursor = [], '*'
    while len(rows) < max_results:
        params = {
            **base_params,
            'cursor': cursor,
            'per-page': min(200, max_results - len(rows)),
        }

        for attempt in range(max_retries):
            try:
                r = requests.get(base_url, params=params, timeout=30)
                r.raise_for_status()
                payload = r.json()
                break
            except requests.RequestException as e:
                if attempt == max_retries - 1:
                    raise RuntimeError(
                        f'OpenAlex 요청 실패 ({attempt+1}/{max_retries}): {e}'
                    ) from e
                time.sleep(retry_backoff ** attempt)

        for w in payload.get('results', []):
            rows.append(_to_row(w))
            if len(rows) >= max_results:
                break

        cursor = (payload.get('meta') or {}).get('next_cursor')
        if not cursor:
            break

    return pd.DataFrame(rows)


def _to_row(w: dict) -> dict:
    """OpenAlex Work → 한 행. 저자 'X 외 N명' + citations_per_year 포함."""
    auths = w.get('authorships', []) or []
    head = ', '.join(a['author']['display_name'] for a in auths[:3])
    authors_str = head + (f' 외 {len(auths)-3}명' if len(auths) > 3 else '')

    year = w.get('publication_year')
    cit = w.get('cited_by_count', 0) or 0
    age = max(2026 - (year or 2026), 1)            # 0년 차 방지
    cit_per_year = round(cit / age, 2)

    return {
        'id': w.get('id'),
        'title': w.get('title'),
        'authors': authors_str,
        'year': year,
        'venue': ((w.get('primary_location') or {}).get('source') or {}).get('display_name'),
        'cited_by_count': cit,
        'citations_per_year': cit_per_year,
        'language': w.get('language'),
        'doi': w.get('doi'),
        'oa_url': (w.get('open_access') or {}).get('oa_url'),
        'abstract': _reconstruct_abstract(w.get('abstract_inverted_index')),
    }


def _reconstruct_abstract(inv_index):
    if not inv_index:
        return None
    positions = []
    for word, idxs in inv_index.items():
        for i in idxs:
            positions.append((i, word))
    positions.sort()
    return ' '.join(w for _, w in positions)


df = search_openalex(QUERY, FROM_YEAR, TO_YEAR, MAX_RESULTS)
print(f'{len(df)} papers fetched')
df.head()


## (선택) 본인 주제로 정밀 재검색

기본 검색 결과가 너무 넓거나 노이즈가 많으면 아래 셀의 옵션을 조정해 다시 호출하세요. 같은 `df`로 덮어쓰면 다음 셀에서 그대로 저장됩니다.

In [ ]:
# 예시 1 — 제목·abstract만 검색 (노이즈 최소화)
# df = search_openalex(QUERY, FROM_YEAR, TO_YEAR, max_results=100,
#                      search_field='title_and_abstract')

# 예시 2 — 영어 논문만, 200건까지
# df = search_openalex(QUERY, FROM_YEAR, TO_YEAR, max_results=200,
#                      search_field='title_and_abstract', language='en')

# 예시 3 — 한국어 + 한국 기관 저자 (국내 연구 발굴)
# df = search_openalex('국부론 자유시장', 2015, 2026, 50,
#                      language='ko', country_code='KR')

# 위 예시 중 하나의 주석을 풀고 실행하면 df가 업데이트됩니다.
df.head()

In [ ]:
out = DATA_DIR / 'papers_raw.csv'
df.to_csv(out, index=False)
print(f'Saved → {out.resolve()}')

## 다음 단계

`02_score_quality.ipynb`을 열어 인용수·연도 기반 정량 점수를 계산하세요.